# Dealing with Counts: Binarization

_Feature Engineering_

**Sometimes only presence matters, not how much — clip a raw count to 0/1.**

A count is a non-negative integer: how many times something happened. Counts are everywhere — plays, clicks, words, visits.

       The trap: counts can be wildly skewed. A few items get astronomically high counts; most get tiny ones. Plotted, the distribution has a long right tail. Fed to a model, the giant values dominate.

---

This notebook builds the idea up one step at a time. Run each cell, read the note above it, watch a skewed count wreck a model and a 0/1 flag rescue it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Load a real, right-skewed count

We pull one genuinely skewed column out of the breast-cancer data: `mean area`. Most tumors are small, but a few are huge, so the values pile up on the left with a long tail to the right. This is exactly the shape of a raw count feature, and it is what we are about to fix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Breast-cancer 'mean area': real, heavily right-skewed (most tumors small, a few huge).
data = load_breast_cancer()
feature_index = data.feature_names.tolist().index("mean area")
area = data.data[:, feature_index]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(area, bins=40, color="#c0392b")
ax[0].axvline(np.median(area), color="black", ls="--", label="median")
ax[0].set_title(f"STEP 1: raw 'mean area' (right-skewed)\nmax={area.max():.0f} = "
                f"{area.max()/np.median(area):.1f}x the median")
ax[0].set_xlabel("mean area")
ax[0].set_ylabel("count")
ax[0].legend()

### Step 2 — Watch the outliers hijack the scale

Standardizing a well-behaved feature lands almost everything in roughly `[-3, +3]`. Here the long right tail throws the top values far past `+3`, so a distance-based model would let this one column drown out every other feature. That is the damage a raw skewed count does.

In [ ]:
# Standardize the raw feature. A well-behaved feature lands roughly in [-3, +3].
z = StandardScaler().fit_transform(area.reshape(-1, 1)).ravel()
n_beyond_3 = (z > 3).sum()

print(f"STEP 2 PROBLEM  raw standardized range = [{z.min():.1f}, {z.max():.1f}]")
print(f"  -> {n_beyond_3} points sit beyond +3 std (top point at +{z.max():.1f}); "
      f"a healthy feature would stop near +3. The tail hijacks the scale.")

### Step 3 — Binarize, and the range comes back under control

Binarizing thresholds the feature at its median: every value becomes a `0` or a `1`. Standardize that flag and its range is bounded and outlier-free — no single point can stretch it anymore. The histogram overlays the blown-out raw values against the tidy binarized ones.

In [ ]:
# Threshold the REAL feature at its median -> 0/1. Now standardized it is bounded.
flag = (area > np.median(area)).astype(int)
zb = StandardScaler().fit_transform(flag.reshape(-1, 1)).ravel()

print(f"STEP 3 FIX      binarized standardized range = [{zb.min():.1f}, {zb.max():.1f}] "
      f"(bounded, outlier-free)")

ax[1].hist(z, bins=40, color="#c0392b", alpha=.7, label="raw (tail to +5)")
ax[1].hist(zb, bins=40, color="#27ae60", alpha=.7, label="binarized (bounded)")
ax[1].set_title("STEP 3: standardized RANGE\nraw blows out vs binarized stays bounded")
ax[1].set_xlabel("standardized value")
ax[1].set_ylabel("count")
ax[1].legend()

### Step 4 — Prove the flag helps a real model

To make the predictive payoff visible we build a small illustrative dataset (fixed seed): a clean informative feature plus a skewed count whose true signal is just "happened at all". The raw count's outliers dominate the k-NN distance and drag accuracy down; the `0/1` flag keeps the signal and removes the outliers. We score both with the identical model and cross-validation so the comparison is honest, and plot how each half of the flag splits by class.

In [ ]:
# ILLUSTRATIVE dataset (fixed rng): a clean informative feature 'good' PLUS a skewed
# COUNT feature whose true signal is "happened at all" but whose raw magnitude has a
# heavy lognormal tail. The raw count's outliers dominate the k-NN distance and drag
# accuracy DOWN; the 0/1 flag fixes it. This mirrors the real 'mean area' skew above.
rng = np.random.default_rng(0)
n = 600
y = rng.integers(0, 2, n)                              # labels
good = y + rng.normal(0, 0.6, n)                       # clean, informative feature
happened = ((y * 0.8 + rng.random(n)) > 0.6).astype(int)   # the TRUE binary signal
count = happened * rng.lognormal(2.0, 1.8, n)          # heavy-tailed raw magnitude

def knn_auc(X):  # same model, same CV everywhere -> honest comparison
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))
    return cross_val_score(pipe, X, y, cv=5, scoring="roc_auc").mean()

X_raw = np.column_stack([good, count])                 # good + raw skewed count
X_bin = np.column_stack([good, (count > 0).astype(int)])  # good + 0/1 flag
raw_auc = knn_auc(X_raw)
eng_auc = knn_auc(X_bin)

# class balance per half of the flag — shows the flag carries real signal
flag_signal = (count > 0).astype(int)
for f in (0, 1):
    in_half = flag_signal == f
    label = "big" if f else "zero"
    print(f"  flag={f} ('{label}'): n={in_half.sum():3d}  "
          f"positive rate = {y[in_half].mean():.2f}")

# bar of the 0/1 split stacked by class
class0_flag0 = ((flag_signal == 0) & (y == 0)).sum()
class1_flag0 = ((flag_signal == 0) & (y == 1)).sum()
class0_flag1 = ((flag_signal == 1) & (y == 0)).sum()
class1_flag1 = ((flag_signal == 1) & (y == 1)).sum()
ax[2].bar([0, 1], [class0_flag0, class0_flag1], color="#2980b9", label="class 0")
ax[2].bar([0, 1], [class1_flag0, class1_flag1],
          bottom=[class0_flag0, class0_flag1], color="#c0392b", label="class 1")
ax[2].set_xticks([0, 1])
ax[2].set_xticklabels(["flag=0", "flag=1"])
ax[2].set_title("STEP 4: 0/1 split (illustrative)\nclass balance per half")
ax[2].legend()
plt.tight_layout()
plt.show()

print(f"STEP 4  k-NN with raw skewed count   ROC-AUC = {raw_auc:.3f}")
print(f"STEP 4  k-NN with binarized 0/1 flag ROC-AUC = {eng_auc:.3f}")
print(f"PROBLEM (raw): {raw_auc:.3f}   ->   FIX (binarized): {eng_auc:.3f}")

## Reference implementation — pandas

### Step 1 — Load the raw play counts and see the skew

The Echo Nest taste profile gives one row per (user, song) with a raw `listen_count`. `describe()` shows the tell-tale shape: a mean of only a few plays but a max in the hundreds — a long right tail driven by super-fans who replay one song over and over.

In [ ]:
import pandas as pd
import numpy as np

# Echo Nest taste profile subset (Million Song Dataset).
# Columns: user (hash), song (hash), listen_count (int).
# Download: github.com/alicezheng/feature-engineering-book  (and the MSD/Echo Nest site)
listen_count = pd.read_csv(
    'train_triplets.txt', sep='\t', header=None,
    names=['user', 'song', 'listen_count']
)

# the raw counts are heavily skewed:
print(listen_count['listen_count'].describe())
# one user can play a single song hundreds of times, while most plays are 1-2:
#   mean  ~ 3, max in the hundreds -> a long right tail

### Step 2 — Binarize: collapse every positive play to 1

For a recommender the signal is simply *did this user listen to this song at all*, not how obsessively. So we clip every positive count to `1` (absence stays `0`). The super-fan's `312` plays collapse to the same `1` as a single play — the outlier's leverage is gone.

In [ ]:
# BINARIZE: for a recommender we only care WHETHER a user listened.
# clip every positive play count to 1; absence stays 0.
listen_count['listen'] = (listen_count['listen_count'] >= 1).astype(int)

# equivalently, with scikit-learn's Binarizer (threshold = 0 keeps >0 as 1):
# from sklearn.preprocessing import binarize
# listen_count['listen'] = binarize(listen_count[['listen_count']], threshold=0.0)

print(listen_count[['listen_count', 'listen']].head(10))
#    listen_count  listen
# 0             1       1
# 1             2       1
# 2             1       1
# 3           312       1     <- the super-fan's count collapses to 1
# 4             1       1

### Step 3 — Keep the robust presence/absence signal

The new `listen` column is the bounded, outlier-free presence flag the book feeds to the recommender, in place of the skewed raw `listen_count`. This is the whole point of binarization: a stable `0/1` signal that no power-user can distort.

In [ ]:
# The 'listen' column is the robust presence/absence signal the book feeds
# to the recommender, instead of the outlier-prone raw 'listen_count'.
print(listen_count[['listen_count', 'listen']].head())

## Visualize the data & results

_On a real right-skewed count-like feature, what does the raw distribution look like, and how does binarizing at the median split the classes?_

### Step 1 — Pick a real right-skewed feature and measure its skew

We reuse the breast-cancer `mean area` column as a stand-in for a count-like feature. Printing its min/median/max and computing the skewness coefficient confirms it: the median sits well below the max and the skew is strongly positive, so the values lean hard to the right.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()                 # 569 real tumor records
names = list(data.feature_names)
X, y = data.data, data.target               # label: 1 = benign, 0 = malignant

# pick a real count-like, right-skewed feature: 'mean area'
col = X[:, names.index('mean area')]
print("min/median/max =", round(col.min(), 1), round(np.median(col), 1), round(col.max(), 1))
# -> 143.5 / 551.1 / 2501.0  (a long right tail)

skew = ((col - col.mean())**3).mean() / col.std()**3
print("skew =", round(float(skew), 2))      # -> 1.64, heavily right-skewed

### Step 2 — Look at the raw histogram counts

Binning the raw feature into 8 buckets makes the tail concrete: the leftmost bins hold most of the rows and the rightmost bins hold only a handful. That pile-up on the left is the distribution shape binarization is meant to tame.

In [ ]:
# raw distribution: histogram counts pile up on the left
counts, edges = np.histogram(col, bins=8)
print("hist counts =", counts.tolist())     # -> [164, 249, 69, 61, 14, 8, 1, 3]

### Step 3 — Binarize at the median and check the split is informative

There is no natural "did it happen" zero here, so we split at the median into a low half and a high half — giving a balanced `0/1` feature. To check the flag actually carries signal, we look at the benign rate inside each half: the small-area half is overwhelmingly benign while the large-area half is mostly malignant, so the single bit separates the classes well.

In [ ]:
# BINARIZE at the median (no natural zero, so split the rows in half)
thr = np.median(col)
b = (col >= thr).astype(int)
print("n(0), n(1) =", int((b == 0).sum()), int((b == 1).sum()))   # -> 284, 285

# is the 0/1 split informative? class balance within each half
for v in (0, 1):
    sel = b == v
    print("bin", v, "-> P(benign) =", round(float(y[sel].mean()), 3))
# bin 0 -> 0.940   (small-area half is mostly benign)
# bin 1 -> 0.316   (large-area half is mostly malignant)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have a "listen_count" column from the Echo Nest data: most values are 1 or 2, but one user played a song 980 times. You are building a song recommender. Should you feed the raw count, and what does binarizing change?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask what the recommender actually needs from this column. — _The training signal is "this user likes this song." Presence of an interaction is the weak positive; the exact replay count is mostly noise._
- Notice the 980 is a power-user artifact. — _A heavy-tailed count lets one row dominate a linear model's loss and distort coefficients — the value is about that user's habit, not preference strength._
- Apply b = 1[x >= 1] to collapse every positive count to 1. — _Binarizing maps 1, 2, and 980 all to 1, removing the skew and the outlier's leverage. The feature becomes "listened at all"._

**Answer:** Don't feed the raw count: it is heavy-tailed and the 980 would dominate. Binarize with threshold 1 — the column becomes a 0/1 "did the user listen at all" flag, which is the robust preference signal the book recommends for an implicit-feedback recommender.

</details>

**Problem 2.** A numeric feature "session_length" has no natural zero event and you still want a single binary split. How do you pick the threshold, and what is the trade-off?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use the median of the feature as the threshold t. — _With no natural "did it happen" cutoff, the median splits the rows into a low half and a high half, so neither bin is empty._
- Apply b = 1[x >= median] to every row. — _This produces a balanced 0/1 feature: roughly half the rows are 1 and half are 0._
- Accept that you have discarded the magnitude. — _Binarizing throws away how far above or below the median each value is. If that magnitude is signal, prefer a log transform or binning instead._

**Answer:** Threshold at the median, giving a balanced 0/1 split (b = 1[x >= median]). The trade-off is that you discard all magnitude information — if the magnitude matters, use a log/power transform or binning rather than binarizing.

</details>